In [ ]:
import nltk
import string
import pandas as pd
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng'], quiet=True)

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

#без лемматизации и стемминга
def preprocess_basic(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    return " ".join([t for t in tokens if t not in stop_words and len(t) > 2])

#стемминг
def preprocess_with_stemming(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    stemmed_tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(stemmed_tokens)

#лемматизация
def preprocess_with_lemmatization(text, only_nouns_adj=False):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    lemmatized_tokens = []
    for token, tag in tagged:
        if token in stop_words or len(token) <= 2:
            continue
        #оставляем только существительные(N) и прилагательные(J)
        if only_nouns_adj:
            if not (tag.startswith('N') or tag.startswith('J')):
                continue
        lemmatized_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))
    return " ".join(lemmatized_tokens)

dataset = load_dataset("emotion", trust_remote_code=True)
train_subset = dataset['train']
test_subset = dataset['test']

y_train = train_subset['label']
y_test = test_subset['label']

results = []

tasks = [
    #Предобработка: без лемм, стемминг + TF-IDF
    ("Без лемм", [preprocess_basic(t) for t in train_subset['text']],
                [preprocess_basic(t) for t in test_subset['text']], [("TF-IDF", TfidfVectorizer())]),

    ("Стемминг", [preprocess_with_stemming(t) for t in train_subset['text']],
                [preprocess_with_stemming(t) for t in test_subset['text']], [("TF-IDF", TfidfVectorizer())]),

    #Лемматизация + 3 варианта векторов
    ("Лемматизация", [preprocess_with_lemmatization(t) for t in train_subset['text']],
                    [preprocess_with_lemmatization(t) for t in test_subset['text']], [
                        ("Binary", CountVectorizer(binary=True)),
                        ("Frequency", CountVectorizer(binary=False)),
                        ("TF-IDF", TfidfVectorizer())
                    ]),

    #Лемматизация (сущ + прил) + 3 варианта векторов
    ("Лемм (Сущ+Прил)", [preprocess_with_lemmatization(t, only_nouns_adj=True) for t in train_subset['text']],
                        [preprocess_with_lemmatization(t, only_nouns_adj=True) for t in test_subset['text']], [
                            ("Binary", CountVectorizer(binary=True)),
                            ("Frequency", CountVectorizer(binary=False)),
                            ("TF-IDF", TfidfVectorizer())
                        ])
]

for label, tr_txt, ts_txt, vectorizers in tasks:
    for vec_name, vec in vectorizers:
        X_train = vec.fit_transform(tr_txt)
        X_test = vec.transform(ts_txt)

        for mod_name, model in [("Decision Tree", DecisionTreeClassifier(random_state=42)),
                                ("Random Forest", RandomForestClassifier(random_state=42))]:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            f1_micro = f1_score(y_test, y_pred, average='micro')
            f1_macro = f1_score(y_test, y_pred, average='macro')
            f1_weighted = f1_score(y_test, y_pred, average='weighted')

            results.append({
                "Task": label,
                "Vector": vec_name,
                "Model": mod_name,
                "F1 Micro": round(f1_micro, 3),
                "F1 Macro": round(f1_macro, 3),
                "F1 Weighted": round(f1_weighted, 3)
            })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


           Task    Vector         Model  F1 Micro  F1 Macro  F1 Weighted
       Без лемм    TF-IDF Decision Tree     0.860     0.807        0.861
       Без лемм    TF-IDF Random Forest     0.885     0.828        0.885
       Стемминг    TF-IDF Decision Tree     0.786     0.720        0.787
       Стемминг    TF-IDF Random Forest     0.849     0.786        0.848
   Лемматизация    Binary Decision Tree     0.830     0.776        0.831
   Лемматизация    Binary Random Forest     0.864     0.803        0.864
   Лемматизация Frequency Decision Tree     0.833     0.778        0.834
   Лемматизация Frequency Random Forest     0.866     0.805        0.865
   Лемматизация    TF-IDF Decision Tree     0.837     0.785        0.837
   Лемматизация    TF-IDF Random Forest     0.859     0.795        0.856
Лемм (Сущ+Прил)    Binary Decision Tree     0.677     0.604        0.675
Лемм (Сущ+Прил)    Binary Random Forest     0.692     0.620        0.689
Лемм (Сущ+Прил) Frequency Decision Tree     0.674  

лучшие результаты получились без лемм + TF-IDF

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score

texts_train = [preprocess_basic(t) for t in train_subset['text']]
texts_test = [preprocess_basic(t) for t in test_subset['text']]

#используем TF-IDF
tfidf_vec = TfidfVectorizer()
X_train_tfidf = tfidf_vec.fit_transform(texts_train)
X_test_tfidf = tfidf_vec.transform(texts_test)

final_tuning = []

#Decision Tree
for depth in [5, 10, 25, 50, 100, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_tfidf, y_train)
    y_pred = dt.predict(X_test_tfidf)

    final_tuning.append({
        "Model": "Decision Tree (TF-IDF)",
        "Parameter": f"max_depth={depth}",
        "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
        "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
        "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
    })

#Random Forest
for n_est in [50, 200, 500]:
    for depth in [50, None]:
        rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
        rf.fit(X_train_tfidf, y_train)
        y_pred = rf.predict(X_test_tfidf)

        final_tuning.append({
            "Model": "Random Forest (TF-IDF)",
            "Parameter": f"n_est={n_est}, depth={depth}",
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
        })

df_final = pd.DataFrame(final_tuning)
print(df_final.to_string(index=False))

                 Model             Parameter  F1 Macro  F1 Micro  F1 Weighted
Decision Tree (TF-IDF)           max_depth=5     0.149     0.359        0.204
Decision Tree (TF-IDF)          max_depth=10     0.178     0.371        0.225
Decision Tree (TF-IDF)          max_depth=25     0.317     0.418        0.303
Decision Tree (TF-IDF)          max_depth=50     0.428     0.483        0.392
Decision Tree (TF-IDF)         max_depth=100     0.603     0.604        0.542
Decision Tree (TF-IDF)        max_depth=None     0.807     0.860        0.861
Random Forest (TF-IDF)    n_est=50, depth=50     0.616     0.718        0.702
Random Forest (TF-IDF)  n_est=50, depth=None     0.825     0.883        0.883
Random Forest (TF-IDF)   n_est=200, depth=50     0.617     0.732        0.713
Random Forest (TF-IDF) n_est=200, depth=None     0.826     0.885        0.884
Random Forest (TF-IDF)   n_est=500, depth=50     0.621     0.736        0.718
Random Forest (TF-IDF) n_est=500, depth=None     0.826     0.887

In [ ]:
import nltk
import string
import pandas as pd
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng'], quiet=True)

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

#без лемматизации и стемминга
def preprocess_basic(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    return " ".join([t for t in tokens if t not in stop_words and len(t) > 2])

#стемминг
def preprocess_with_stemming(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    stemmed_tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(stemmed_tokens)

#лемматизация
def preprocess_with_lemmatization(text, only_nouns_adj=False):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    lemmatized_tokens = []
    for token, tag in tagged:
        if token in stop_words or len(token) <= 2:
            continue
        #оставляем только существительные(N) и прилагательные(J)
        if only_nouns_adj:
            if not (tag.startswith('N') or tag.startswith('J')):
                continue
        lemmatized_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))
    return " ".join(lemmatized_tokens)

dataset = load_dataset("emotion")
train_subset = dataset['train']
test_subset = dataset['test']

y_train = train_subset['label']
y_test = test_subset['label']

results = []

tasks = [
    #Предобработка: без лемм, стемминг + TF-IDF
    ("Без лемм", [preprocess_basic(t) for t in train_subset['text']],
                [preprocess_basic(t) for t in test_subset['text']], [("TF-IDF", TfidfVectorizer())]),

    ("Стемминг", [preprocess_with_stemming(t) for t in train_subset['text']],
                [preprocess_with_stemming(t) for t in test_subset['text']], [("TF-IDF", TfidfVectorizer())]),

    #Лемматизация + 3 варианта векторов
    ("Лемматизация", [preprocess_with_lemmatization(t) for t in train_subset['text']],
                    [preprocess_with_lemmatization(t) for t in test_subset['text']], [
                        ("Binary", CountVectorizer(binary=True)),
                        ("Frequency", CountVectorizer(binary=False)),
                        ("TF-IDF", TfidfVectorizer())
                    ]),

    #Лемматизация (сущ + прил) + 3 варианта векторов
    ("Лемм (Сущ+Прил)", [preprocess_with_lemmatization(t, only_nouns_adj=True) for t in train_subset['text']],
                        [preprocess_with_lemmatization(t, only_nouns_adj=True) for t in test_subset['text']], [
                            ("Binary", CountVectorizer(binary=True)),
                            ("Frequency", CountVectorizer(binary=False)),
                            ("TF-IDF", TfidfVectorizer())
                        ])
]

for label, tr_txt, ts_txt, vectorizers in tasks:
    for vec_name, vec in vectorizers:
        X_train = vec.fit_transform(tr_txt)
        X_test = vec.transform(ts_txt)

        models_to_test = [
            ("GBM", GradientBoostingClassifier(random_state=42)),
            ("AdaBoost", AdaBoostClassifier(random_state=42))
        ]

        for mod_name, model in models_to_test:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            f1_micro = f1_score(y_test, y_pred, average='micro')
            f1_macro = f1_score(y_test, y_pred, average='macro')
            f1_weighted = f1_score(y_test, y_pred, average='weighted')

            results.append({
                "Task": label,
                "Vector": vec_name,
                "Model": mod_name,
                "F1 Micro": round(f1_micro, 3),
                "F1 Macro": round(f1_macro, 3),
                "F1 Weighted": round(f1_weighted, 3)
            })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

           Task    Vector    Model  F1 Micro  F1 Macro  F1 Weighted
       Без лемм    TF-IDF      GBM     0.846     0.801        0.846
       Без лемм    TF-IDF AdaBoost     0.348     0.090        0.186
       Стемминг    TF-IDF      GBM     0.796     0.761        0.797
       Стемминг    TF-IDF AdaBoost     0.354     0.104        0.195
   Лемматизация    Binary      GBM     0.800     0.767        0.801
   Лемматизация    Binary AdaBoost     0.350     0.096        0.189
   Лемматизация Frequency      GBM     0.800     0.767        0.801
   Лемматизация Frequency AdaBoost     0.350     0.096        0.189
   Лемматизация    TF-IDF      GBM     0.801     0.767        0.802
   Лемматизация    TF-IDF AdaBoost     0.351     0.097        0.192
Лемм (Сущ+Прил)    Binary      GBM     0.680     0.635        0.678
Лемм (Сущ+Прил)    Binary AdaBoost     0.352     0.098        0.193
Лемм (Сущ+Прил) Frequency      GBM     0.683     0.637        0.680
Лемм (Сущ+Прил) Frequency AdaBoost     0.352    

Градиентный бустинг показал лучший результат при Без лемм + TF-IDF, для адаптивного будем использовать стемминг + TF-IDF

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score

#для GBM (Без лемм + TF-IDF)
texts_train_gbm = [preprocess_basic(t) for t in train_subset['text']]
texts_test_gbm = [preprocess_basic(t) for t in test_subset['text']]

tfidf_gbm = TfidfVectorizer()
X_train_gbm = tfidf_gbm.fit_transform(texts_train_gbm)
X_test_gbm = tfidf_gbm.transform(texts_test_gbm)

#для AdaBoost (Стемминг + TF-IDF)
texts_train_ada = [preprocess_with_stemming(t) for t in train_subset['text']]
texts_test_ada = [preprocess_with_stemming(t) for t in test_subset['text']]

tfidf_ada = TfidfVectorizer(max_features=5000)
X_train_ada = tfidf_ada.fit_transform(texts_train_ada)
X_test_ada = tfidf_ada.transform(texts_test_ada)

y_train = np.array(train_subset['label'])
y_test = np.array(test_subset['label'])

final_tuning = []

#Gradient Boosting
for n_est in [100, 200]:
    for depth in [3, 5]:
        gbm = GradientBoostingClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
        gbm.fit(X_train_gbm, y_train)
        y_pred = gbm.predict(X_test_gbm)

        final_tuning.append({
            "Model": "GBM (No Lemm + TF-IDF)",
            "Parameter": f"n_est={n_est}, depth={depth}",
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
        })

#AdaBoost
for n_est in [50, 100]:
    for base_depth in [5, 10]:
        ada = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=base_depth, random_state=42),
            n_estimators=n_est,
            random_state=42
        )
        ada.fit(X_train_ada, y_train)
        y_pred = ada.predict(X_test_ada)

        final_tuning.append({
            "Model": "AdaBoost (Stem + TF-IDF)",
            "Parameter": f"n_est={n_est}, base_depth={base_depth}",
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
        })

df_final = pd.DataFrame(final_tuning)
print(df_final.to_string(index=False))

                   Model                Parameter  F1 Macro  F1 Micro  F1 Weighted
  GBM (No Lemm + TF-IDF)       n_est=100, depth=3     0.801     0.846        0.846
  GBM (No Lemm + TF-IDF)       n_est=100, depth=5     0.821     0.874        0.874
  GBM (No Lemm + TF-IDF)       n_est=200, depth=3     0.830     0.877        0.879
  GBM (No Lemm + TF-IDF)       n_est=200, depth=5     0.834     0.884        0.885
AdaBoost (Stem + TF-IDF)   n_est=50, base_depth=5     0.215     0.364        0.249
AdaBoost (Stem + TF-IDF)  n_est=50, base_depth=10     0.361     0.377        0.322
AdaBoost (Stem + TF-IDF)  n_est=100, base_depth=5     0.277     0.385        0.304
AdaBoost (Stem + TF-IDF) n_est=100, base_depth=10     0.419     0.453        0.385


задания 6, 7, 8

In [ ]:
import nltk
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

def preprocess_selective(text, allowed_pos=['N', 'J']):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)

    selected_tokens = []
    for token, tag in tagged:
        if token in stop_words or len(token) <= 2:
            continue

        if any(tag.startswith(pos) for pos in allowed_pos):
            selected_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))

    return " ".join(selected_tokens)

pos_tasks = [
    ("Только Сущ+Прил", ['N', 'J']),
    ("Сущ+Прил+Глаголы", ['N', 'J', 'V']),
    ("Только Глаголы", ['V'])
]

lsa_results = []

for label, pos_list in pos_tasks:
    train_txt = [preprocess_selective(t, allowed_pos=pos_list) for t in train_subset['text']]
    test_txt = [preprocess_selective(t, allowed_pos=pos_list) for t in test_subset['text']]

    tfidf = TfidfVectorizer()
    X_train_tfidf = tfidf.fit_transform(train_txt)
    X_test_tfidf = tfidf.transform(test_txt)

    for n_comp in [50, 100]:
        if X_train_tfidf.shape[1] <= n_comp: continue

        svd = TruncatedSVD(n_components=n_comp, random_state=42)
        X_train_lsa = svd.fit_transform(X_train_tfidf)
        X_test_lsa = svd.transform(X_test_tfidf)

        #используем GBM
        model = GradientBoostingClassifier(random_state=42)
        model.fit(X_train_lsa, y_train)
        y_pred = model.predict(X_test_lsa)

        lsa_results.append({
            "POS": label,
            "LSA Comp": n_comp,
            "F1 Micro": round(f1_score(y_test, y_pred, average='micro'), 3),
            "F1 Macro": round(f1_score(y_test, y_pred, average='macro'), 3),
            "F1 Weighted": round(f1_score(y_test, y_pred, average='weighted'), 3)
        })

df_lsa = pd.DataFrame(lsa_results)
print(df_lsa.to_string(index=False))

             POS  LSA Comp  F1 Micro  F1 Macro  F1 Weighted
 Только Сущ+Прил        50     0.491     0.329        0.448
 Только Сущ+Прил       100     0.551     0.422        0.521
Сущ+Прил+Глаголы        50     0.472     0.294        0.425
Сущ+Прил+Глаголы       100     0.509     0.368        0.472
  Только Глаголы        50     0.426     0.290        0.373
  Только Глаголы       100     0.434     0.306        0.385


In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

#без лемм + TF-IDF
#подготовим данные
texts_train_lsa = [preprocess_basic(t) for t in train_subset['text']]
texts_test_lsa = [preprocess_basic(t) for t in test_subset['text']]

lsa_tuning = []

#n_components задает длину вектора после LSA
for n_topics in [10, 50, 100, 200, 500]:

    lsa_pipe = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('lsa', TruncatedSVD(n_components=n_topics, random_state=42)),
        ('gbm', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42))
    ])

    lsa_pipe.fit(texts_train_lsa, y_train)
    y_pred = lsa_pipe.predict(texts_test_lsa)

    #метрики
    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    lsa_tuning.append({
        "Model": "GBM + LSA",
        "Topics (n_comp)": n_topics,
        "F1 Micro": round(f1_micro, 3),
        "F1 Macro": round(f1_macro, 3),
        "F1 Weighted": round(f1_weighted, 3)
    })

df_lsa = pd.DataFrame(lsa_tuning)
print("\n Результаты LSA эксперимента")
print(df_lsa.to_string(index=False))


 Результаты LSA эксперимента
    Model  Topics (n_comp)  F1 Micro  F1 Macro  F1 Weighted
GBM + LSA               10     0.389     0.189        0.322
GBM + LSA               50     0.474     0.326        0.434
GBM + LSA              100     0.564     0.415        0.534
GBM + LSA              200     0.662     0.556        0.645
GBM + LSA              500     0.769     0.702        0.763


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

y_train = np.array(train_subset['label'])
y_test = np.array(test_subset['label'])

texts_basic = [preprocess_basic(t) for t in train_subset['text']]
texts_stem = [preprocess_with_stemming(t) for t in train_subset['text']]

#Частотная векторизация (для Decision Tree)
vec_freq = CountVectorizer(binary=False)
X_train_freq = vec_freq.fit_transform(texts_basic)

#Бинарная векторизация (для Random Forest)
vec_bin = CountVectorizer(binary=True)
X_train_bin = vec_bin.fit_transform(texts_basic)

#TF-IDF + векторизация (для GBM)
vec_tfidf = TfidfVectorizer()
X_train_gbm = vec_tfidf.fit_transform(texts_basic)

#TF-IDF + стем (для AdaBoost)
vec_ada = TfidfVectorizer(max_features=5000)
X_train_ada = vec_ada.fit_transform(texts_stem)

auto_tuning_results = []

#Decision Tree
dt_params = {
    'max_depth': [10, 50, None],
    'min_samples_leaf': [1, 5]
}
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params, cv=3, scoring='f1_micro', n_jobs=-1
)
dt_grid.fit(X_train_freq, y_train)
auto_tuning_results.append({
    "Model": "Decision Tree",
    "Best Params": dt_grid.best_params_,
    "Best F1 Micro": round(dt_grid.best_score_, 3)
})

#Random Forest
rf_params = {
    'n_estimators': [100, 300],
    'max_depth': [50, None],
    'min_samples_split': [2, 10]
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params, n_iter=4, cv=3, scoring='f1_micro', random_state=42, n_jobs=-1
)
rf_search.fit(X_train_bin, y_train)
auto_tuning_results.append({
    "Model": "Random Forest",
    "Best Params": rf_search.best_params_,
    "Best F1 Micro": round(rf_search.best_score_, 3)
})

#Gradient Boosting
gbm_params = {
    'learning_rate': [0.1],
    'n_estimators': [100, 250],
    'max_depth': [3, 5]
}
gbm_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gbm_params, cv=3, scoring='f1_micro', n_jobs=-1
)
gbm_grid.fit(X_train_gbm, y_train)
auto_tuning_results.append({
    "Model": "GBM",
    "Best Params": gbm_grid.best_params_,
    "Best F1 Micro": round(gbm_grid.best_score_, 3)
})

#AdaBoost
ada_params = {
    'estimator': [DecisionTreeClassifier(max_depth=5), DecisionTreeClassifier(max_depth=10)],
    'n_estimators': [50, 100],
    'learning_rate': [0.1]
}
ada_grid = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    ada_params, cv=3, scoring='f1_micro', n_jobs=-1
)
ada_grid.fit(X_train_ada, y_train)
auto_tuning_results.append({
    "Model": "AdaBoost",
    "Best Params": ada_grid.best_params_,
    "Best F1 Micro": round(ada_grid.best_score_, 3)
})

df_auto = pd.DataFrame(auto_tuning_results)
print("\nРезультаты автоматической настройки гиперпараметров:")
print(df_auto.to_string(index=False))


Результаты автоматической настройки гиперпараметров:
        Model                                                                                    Best Params  Best F1 Micro
Decision Tree                                                     {'max_depth': None, 'min_samples_leaf': 1}          0.877
Random Forest                              {'n_estimators': 300, 'min_samples_split': 10, 'max_depth': None}          0.886
          GBM                                    {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 250}          0.886
     AdaBoost {'estimator': DecisionTreeClassifier(max_depth=10), 'learning_rate': 0.1, 'n_estimators': 100}          0.361


In [ ]:
import pandas as pd
import numpy as np
import string
import nltk
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import f1_score

nltk.download(['punkt', 'punkt_tab', 'wordnet', 'omw-1.4', 'stopwords', 'averaged_perceptron_tagger_eng'], quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def preprocess_basic(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    return " ".join([t for t in tokens if t not in stop_words and len(t) > 2])

dataset = load_dataset("emotion", trust_remote_code=True)
train_subset = dataset['train']

texts_cv = [preprocess_basic(t) for t in train_subset['text']]
y_cv = np.array(train_subset['label'])

tfidf_cv = TfidfVectorizer()
X_cv = tfidf_cv.fit_transform(texts_cv)

models_to_compare = [
    ("Decision Tree", DecisionTreeClassifier(max_depth=None, random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1))
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_cv_results = []

for name, model in models_to_compare:
    cv_output = cross_validate(
        model,
        X_cv,
        y_cv,
        cv=kf,
        scoring=['f1_micro', 'f1_macro'],
        return_train_score=False
    )

    all_cv_results.append({
        "Model": name,
        "F1 Micro (Mean)": round(cv_output['test_f1_micro'].mean(), 3),
        "F1 Micro (Std)": round(cv_output['test_f1_micro'].std(), 4),
        "F1 Macro (Mean)": round(cv_output['test_f1_macro'].mean(), 3)
    })

print("\n Сравнение моделей с использованием Кросс-Валидации (CV=5)")
df_comparison = pd.DataFrame(all_cv_results)
print(df_comparison.to_string(index=False))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



 Сравнение моделей с использованием Кросс-Валидации (CV=5)
        Model  F1 Micro (Mean)  F1 Micro (Std)  F1 Macro (Mean)
Decision Tree            0.866          0.0069            0.827
Random Forest            0.881          0.0024            0.844
